# Real-Time Object Detection

### From Images To Webcam

In p03 we ran YOLO on a single image.

A webcam is just a stream of images — one frame at a time.

```
Frame 1 → YOLO → detections
Frame 2 → YOLO → detections
Frame 3 → YOLO → detections
...
```

The pipeline is identical — we just wrap it in a loop.

### Frame Skipping Optimization

Running YOLO on every frame is unnecessary:
- scene barely changes between consecutive frames
- reuse previous detection for 4 frames
- recalculate only on the 5th

Result:
```
Without skipping:  30 model calls/sec
With skipping:     6 model calls/sec  →  5x less work
```

We do NOT downscale before passing to YOLO:
- YOLOv8 internally resizes to 640×640 anyway
- pre-downscaling gives minimal speed benefit
- can actually hurt accuracy

### Setup

In [ ]:
from ultralytics import YOLO
import cv2
import time

### Constants

In [ ]:
DETECT_EVERY = 5      # run YOLO every N frames
CONF_THRESHOLD = 0.5   # minimum confidence to show a detection

### Load Model

In [ ]:
model = YOLO('yolov8n.pt')

### Detection Function

In [ ]:
def detect(frame):
    
    results = model(frame, conf=CONF_THRESHOLD, verbose=False)
    result  = results[0]
    
    detections = []
    
    for box in result.boxes:
        
        class_id   = int(box.cls.item())
        label      = model.names[class_id]
        confidence = box.conf.item()
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        
        detections.append((label, confidence, (x1, y1, x2, y2)))
    
    return detections

Each detection is stored as:

```python
(label, confidence, (x1, y1, x2, y2))
```

Example:

```python
('person', 0.91, (100, 50, 300, 400))
('car',    0.87, (200, 150, 500, 380))
```

### Draw Function

In [ ]:
def draw(frame, detections):
    
    for label, confidence, (x1, y1, x2, y2) in detections:
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        text = f"{label} {confidence:.2f}"
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    return frame

### Webcam Loop

In [ ]:
cap = cv2.VideoCapture(0)

stored_detections = []
frame_count       = 0
fps               = 0
prev_time         = time.time()
display_fps       = "FPS: --"

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Detection
    if frame_count % DETECT_EVERY == 0:
        stored_detections = detect(frame)

    # Draw
    frame = draw(frame, stored_detections)

    # FPS
    new_time = time.time()
    fps = 0.9 * fps + 0.1 * (1 / (new_time - prev_time))
    prev_time = new_time

    if frame_count % DETECT_EVERY == 0:
        display_fps = f"FPS: {int(fps)}"

    if not stored_detections:
        display_fps = "Idle"

    cv2.putText(frame, display_fps, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(frame, f"Objects: {len(stored_detections)}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Display
    cv2.imshow("YOLOv8 Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

#### `if frame_count % DETECT_EVERY == 0`

Run YOLO only on every 5th frame. All other frames reuse `stored_detections`.

#### `fps = 0.9 * fps + 0.1 * (1 / (new_time - prev_time))`

Smoothed FPS — same formula from Phase 5. Prevents the counter from jumping around every frame.

#### `if not stored_detections`

If nothing is detected in the scene, show Idle instead of FPS — signals the model is barely working.

### Optional — Class Filtering

YOLO detects all 80 COCO classes by default.

If we only care about specific objects, we can filter detections after inference:

```python
WATCH_CLASSES = ['person', 'car', 'bicycle']

def detect_filtered(frame):
    detections = detect(frame)
    return [(l, c, b) for l, c, b in detections if l in WATCH_CLASSES]
```

This doesn't make YOLO faster — the model still processes all 80 classes internally.
It just controls what gets drawn on screen.

Useful when:

- Scene is cluttered with irrelevant objects
- Building a system that only cares about specific targets (e.g. people counter, vehicle tracker)

### Summary

| Concept | What It Does |
| ------- | ------------ |
| `model(frame, verbose=False)` | Run detection, suppress console output |
| `DETECT_EVERY` | Frame skipping — run model every N frames |
| `stored_detections` | Reused detections on skipped frames |
| Smoothed FPS | `0.9 * old + 0.1 * new` — prevents jumpy counter |
| Class filtering | Post-inference filter — controls what gets drawn |

Next up — object tracking. 

Right now boxes appear and disappear every frame with no identity. Tracking assigns each object a persistent ID so we can follow it across frames.